In [ ]:
import pandas as pd

df = pd.read_csv("Sector_Mapping.csv")

In [ ]:
sectors = df['Sector'].unique().tolist()
sectors

In [13]:
from collections import defaultdict

sub_sectors = defaultdict(list)

for sector in sectors:
    sub_sectors[sector].extend(df[df["Sector"]==sector]['Sub-Sector'].to_numpy().tolist())

In [ ]:
sub_sectors
# df[df["Sector"]=="Industry Development"]['Sub-Sector'].to_numpy().tolist()

In [16]:
import json

with open("sub_sectors.json","w+") as file:
    json.dump(sub_sectors,file)

In [20]:
df_positions = pd.read_csv("job positions - Tibarek Final.csv")
df_positions.head()

,informal work in eng,Profession in Amharic,professional/ formal,sector,sub sector
0,Digital Media Marketer,ዲጂታል ሚዲያ ገበያተኛ,P,Business,NaN
1,Market Strategist,ገበያ ስትራቴጂስት,P,Business,NaN
2,Junior Risk Officer,ጁኒየር አደጋ መኮንን,P,Finance,NaN
3,Senior Customer Service Officer,ከፍተኛ የደንበኞች አገልግሎት ኦፊሰር,P,Business,NaN
4,Waiteress/waiter,አስተናጋጅ,P,Service,NaN


In [22]:
positions = df_positions['informal work in eng']
df_positions

,informal work in eng,Profession in Amharic,professional/ formal,sector,sub sector
0,Digital Media Marketer,ዲጂታል ሚዲያ ገበያተኛ,P,Business,NaN
1,Market Strategist,ገበያ ስትራቴጂስት,P,Business,NaN
2,Junior Risk Officer,ጁኒየር አደጋ መኮንን,P,Finance,NaN
3,Senior Customer Service Officer,ከፍተኛ የደንበኞች አገልግሎት ኦፊሰር,P,Business,NaN
4,Waiteress/waiter,አስተናጋጅ,P,Service,NaN
...,...,...,...,...,...
4130,Tiktok account content creation assistance,የቲክቶክ መለያ ይዘት መፍጠር እገዛ,U,NaN,NaN
4131,Tiktok Account Manager,Tiktok መለያ አስተዳዳሪ,P,NaN,NaN
4132,Tiktok advertiser's,የቲክቶክ አስተዋዋቂ,U,NaN,NaN
4133,TikTok and Advertising,TikTok እና ማስታወቂያ,U,NaN,NaN


In [ ]:
# pipeline: map positions -> sector & subsector

import json
import pandas as pd
import requests
import time
from dotenv import load_dotenv
import os
load_dotenv(override=True)


API_KEY = os.getenv("DEEPSEEK_API_KEY",None)
API_URL = "https://api.deepseek.com/v1/chat/completions"


def classifier(positions, sector_sub_sectors):
    """
    Send batch of positions to DeepSeek and return parsed JSON result
    """

    prompt = f"""
You are a classifier that maps job positions to sector and subsector.

Here is the sector and subsector JSON:
{json.dumps(sector_sub_sectors, indent=2)}

Rules:
- Choose ONLY ONE best match
- If unsure, return "None"
- Response MUST be valid JSON only
- No explanation text

Example:
{{
  "Branch Manager": {{
    "sector": "Business",
    "sub_sector": "Wholesale business in specialized stores"
  }}
}}
"""

    try:
        response = requests.post(
            API_URL,
            headers={
                "Authorization": f"Bearer {API_KEY}",
                "Content-Type": "application/json"
            },
            json={
                "model": "deepseek-reasoner",
                "messages": [
                    {"role": "system", "content": prompt},
                    {"role": "user", "content": json.dumps(positions)}
                ],
                "temperature": 0
            }
        )

        result = response.json()

        content = result["choices"][0]["message"]["content"]


        try:
            parsed = json.loads(content)
        except:

            start = content.find("{")
            end = content.rfind("}") + 1
            parsed = json.loads(content[start:end])

        return parsed

    except Exception as e:
        print("Error:", e)
        return {}


def get_unmapped_positions(df):
    """
    Filter only rows that need mapping
    """
    # Ensure columns exist
    if "sector" not in df.columns:
        df["sector"] = None
    if "sub sector" not in df.columns:
        df["sub sector"] = None

    # Unmapped = sector or sub sector is missing/empty
    mask = (
        df["sector"].isna() | (df["sector"] == "") |
        df["sub sector"].isna() | (df["sub sector"] == "")
    )

    unmapped_df = df[mask]

    # Remove duplicates to reduce token usage
    positions = unmapped_df["informal work in eng"].dropna().unique().tolist()

    return positions, mask

def apply_results_to_df(df, results):
    """
    Update dataframe with sector & subsector
    """

    df["sector"] = None
    df["sub sector"] = None

    for idx, row in df.iterrows():
        position = row["informal work in eng"]

        if position in results:
            df.at[idx, "sector"] = results[position].get("sector")
            df.at[idx, "sub sector"] = results[position].get("sub_sector")

    return df


def batch_process(positions, sector_json, batch_size=20):
    """
    Production batching to reduce token usage
    """

    all_results = {}

    for i in range(0, len(positions), batch_size):
        batch = positions[i:i + batch_size]

        print(f"Processing batch {i // batch_size + 1}...")

        result = classifier(batch, sector_json)

        all_results.update(result)

        # avoid rate limit
        time.sleep(1)

    return all_results


def sector_subsector_mapper(test_mode=True):
    # load sector-subsector
    with open("sub_sectors.json", "r") as file:
        sub_sectors_json = json.load(file)

    # load positions
    df_positions = pd.read_csv("job positions - Tibarek Final.csv")
    df_mapped = None
    if os.path.exists("output_with_sectors.csv"):
        df_mapped = pd.read_csv("output_with_sectors.csv")
    #positions = df_positions['informal work in eng'].dropna().tolist()
    positions, mask = get_unmapped_positions(df_mapped or df_positions)

    if test_mode:
        print("Running TEST MODE (first 10 positions)")
        results = classifier(positions[:10], sub_sectors_json)
    else:
        print("Running PRODUCTION MODE (batch processing)")
        results = batch_process(positions, sub_sectors_json, batch_size=25)

    # apply results
    df_positions = apply_results_to_df(df_positions, results)

    # save output
    df_positions.to_csv("output_with_sectors.csv", index=False)

    print("✅ Processing completed. Output saved to output_with_sectors.csv")


if __name__ == "__main__":
    # change to False for production
    sector_subsector_mapper(test_mode=True)

In [31]:
b = pd.DataFrame()
a = 1
b.empty or a

True